### Depth-First Search, LB=T

In [36]:
jobs = [1,2,3,4,5]
processing = {1:4, 2:3, 3:7, 4:2, 5:2}
due = {1:5, 2:6, 3:8, 4:8, 5:17}

def complement(S):
    return [j for j in jobs if j not in S]

def P(S):
    return sum(processing[i] for i in S)

# 실제 순서(order)에 대한 진짜 비용 ΣT_i
def true_cost(order):
    C = 0
    tot = 0
    for j in order:
        C += processing[j]
        tot += max(0, C - due[j])
    return tot

# (수정된) suffix-기반 LB1: 남은 앞부분 Sbar를 다 처리하고(B = P(Sbar)) 난 뒤,
# 고정된 suffix를 실제 순서(reversed(S))로 붙였을 때 생기는 T만 더한 값
def LB1_suffix(S):
    Sbar = complement(S)
    C = P(Sbar)          # suffix가 시작할 '가장 이른' 시작시각
    lb = 0
    for j in reversed(S):  # suffix의 실제 처리 순서
        C += processing[j]
        lb += max(0, C - due[j])
    return lb  # 앞부분은 아직 미정이므로 0 이상 -> 이게 안전한 LB

# (수정된) LB2: LB1_suffix + 앞부분(Sbar)에서 최소 하나는 납기초과가 날 수 있는 "한 방" 보정
def LB2_suffix(S):
    base = LB1_suffix(S)
    Sbar = complement(S)
    if not Sbar: 
        return base
    extra = min(max(0, P(Sbar) - due[k]) for k in Sbar)
    return base + extra

def dfs_backtrack(S, depth=0, mode="LB1", step=[1], IC=[float("inf")], best=[None]):
    # LB 선택
    lb = LB1_suffix(S) if mode == "LB1" else LB2_suffix(S)

    indent = "    " * depth
    seq = "".join(str(j) for j in reversed(S))
    print(f"{indent}[Step {step[0]}] 뒤쪽에 [{'?'*(len(jobs)-len(S))}{seq}] 배치, 남은 {complement(S)}, LB={lb}, IC={IC[0]}")
    step[0]+=1

    # 모든 작업 배치(리프): 반드시 '진짜 비용'으로 평가
    if len(S) == len(jobs):
        order = list(reversed(S))           # 전체 순서 (앞→뒤)
        cost = true_cost(order)             # 여기서 LB 쓰지 말 것!
        if cost < IC[0]:
            IC[0] = cost
            best[0] = order[:]
            print(f"{indent}   --> 모든 작업 배치완료! 순서={order}, IC 갱신: {IC[0]}")
        return

    # 가지치기
    if lb >= IC[0]:
        print(f"{indent}   --> 가지치기 (LB ≥ IC)")
        return

    # 아직 배치 안 된 것 중 하나를 뒤(오른쪽)로 보냄
    for j in complement(S):
        dfs_backtrack(S+[j], depth+1, mode, step, IC, best)

# 실행 예시
print("\n=== (a) DFS with LB1 ===")
dfs_backtrack([], mode="LB1", step=[1], IC=[float("inf")], best=[None])


=== (a) DFS with LB1 ===
[Step 1] 뒤쪽에 [?????] 배치, 남은 [1, 2, 3, 4, 5], LB=0, IC=inf
    [Step 2] 뒤쪽에 [????1] 배치, 남은 [2, 3, 4, 5], LB=13, IC=inf
        [Step 3] 뒤쪽에 [???21] 배치, 남은 [3, 4, 5], LB=21, IC=inf
            [Step 4] 뒤쪽에 [??321] 배치, 남은 [4, 5], LB=24, IC=inf
                [Step 5] 뒤쪽에 [?4321] 배치, 남은 [5], LB=24, IC=inf
                    [Step 6] 뒤쪽에 [54321] 배치, 남은 [], LB=24, IC=inf
                       --> 모든 작업 배치완료! 순서=[5, 4, 3, 2, 1], IC 갱신: 24
                [Step 7] 뒤쪽에 [?5321] 배치, 남은 [4], LB=24, IC=24
                   --> 가지치기 (LB ≥ IC)
            [Step 8] 뒤쪽에 [??421] 배치, 남은 [3, 5], LB=24, IC=24
               --> 가지치기 (LB ≥ IC)
            [Step 9] 뒤쪽에 [??521] 배치, 남은 [3, 4], LB=21, IC=24
                [Step 10] 뒤쪽에 [?3521] 배치, 남은 [4], LB=22, IC=24
                    [Step 11] 뒤쪽에 [43521] 배치, 남은 [], LB=22, IC=24
                       --> 모든 작업 배치완료! 순서=[4, 3, 5, 2, 1], IC 갱신: 22
                [Step 12] 뒤쪽에 [?4521] 배치, 남은 [3], LB=22, IC=22
                  

### Depth-First Search, LB=T+min

In [37]:
jobs = [1,2,3,4,5]
processing = {1:4, 2:3, 3:7, 4:2, 5:2}
due = {1:5, 2:6, 3:8, 4:8, 5:17}

def complement(S):
    return [j for j in jobs if j not in S]

def P(S):
    return sum(processing[i] for i in S)

# 실제 순서(order)에 대한 진짜 비용 ΣT_i
def true_cost(order):
    C = 0
    tot = 0
    for j in order:
        C += processing[j]
        tot += max(0, C - due[j])
    return tot

# (수정된) suffix-기반 LB1: 남은 앞부분 Sbar를 다 처리하고(B = P(Sbar)) 난 뒤,
# 고정된 suffix를 실제 순서(reversed(S))로 붙였을 때 생기는 T만 더한 값
def LB1_suffix(S):
    Sbar = complement(S)
    C = P(Sbar)          # suffix가 시작할 '가장 이른' 시작시각
    lb = 0
    for j in reversed(S):  # suffix의 실제 처리 순서
        C += processing[j]
        lb += max(0, C - due[j])
    return lb  # 앞부분은 아직 미정이므로 0 이상 -> 이게 안전한 LB

# (수정된) LB2: LB1_suffix + 앞부분(Sbar)에서 최소 하나는 납기초과가 날 수 있는 "한 방" 보정
def LB2_suffix(S):
    base = LB1_suffix(S)
    Sbar = complement(S)
    if not Sbar: 
        return base
    extra = min(max(0, P(Sbar) - due[k]) for k in Sbar)
    return base + extra

def dfs_backtrack(S, depth=0, mode="LB1", step=[1], IC=[float("inf")], best=[None]):
    # LB 선택
    lb = LB1_suffix(S) if mode == "LB1" else LB2_suffix(S)

    indent = "    " * depth
    seq = "".join(str(j) for j in reversed(S))
    print(f"{indent}[Step {step[0]}] 뒤쪽에 [{'?'*(len(jobs)-len(S))}{seq}] 배치, 남은 {complement(S)}, LB={lb}, IC={IC[0]}")
    step[0]+=1

    # 모든 작업 배치(리프): 반드시 '진짜 비용'으로 평가
    if len(S) == len(jobs):
        order = list(reversed(S))           # 전체 순서 (앞→뒤)
        cost = true_cost(order)             # 여기서 LB 쓰지 말 것!
        if cost < IC[0]:
            IC[0] = cost
            best[0] = order[:]
            print(f"{indent}   --> 모든 작업 배치완료! 순서={order}, IC 갱신: {IC[0]}")
        return

    # 가지치기
    if lb >= IC[0]:
        print(f"{indent}   --> 가지치기 (LB ≥ IC)")
        return

    # 아직 배치 안 된 것 중 하나를 뒤(오른쪽)로 보냄
    for j in complement(S):
        dfs_backtrack(S+[j], depth+1, mode, step, IC, best)

# 실행 예시
print("\n=== (b) DFS with LB2 ===")
dfs_backtrack([], mode="LB2", step=[1], IC=[float("inf")], best=[None])


=== (b) DFS with LB2 ===
[Step 1] 뒤쪽에 [?????] 배치, 남은 [1, 2, 3, 4, 5], LB=1, IC=inf
    [Step 2] 뒤쪽에 [????1] 배치, 남은 [2, 3, 4, 5], LB=13, IC=inf
        [Step 3] 뒤쪽에 [???21] 배치, 남은 [3, 4, 5], LB=21, IC=inf
            [Step 4] 뒤쪽에 [??321] 배치, 남은 [4, 5], LB=24, IC=inf
                [Step 5] 뒤쪽에 [?4321] 배치, 남은 [5], LB=24, IC=inf
                    [Step 6] 뒤쪽에 [54321] 배치, 남은 [], LB=24, IC=inf
                       --> 모든 작업 배치완료! 순서=[5, 4, 3, 2, 1], IC 갱신: 24
                [Step 7] 뒤쪽에 [?5321] 배치, 남은 [4], LB=24, IC=24
                   --> 가지치기 (LB ≥ IC)
            [Step 8] 뒤쪽에 [??421] 배치, 남은 [3, 5], LB=24, IC=24
               --> 가지치기 (LB ≥ IC)
            [Step 9] 뒤쪽에 [??521] 배치, 남은 [3, 4], LB=22, IC=24
                [Step 10] 뒤쪽에 [?3521] 배치, 남은 [4], LB=22, IC=24
                    [Step 11] 뒤쪽에 [43521] 배치, 남은 [], LB=22, IC=24
                       --> 모든 작업 배치완료! 순서=[4, 3, 5, 2, 1], IC 갱신: 22
                [Step 12] 뒤쪽에 [?4521] 배치, 남은 [3], LB=22, IC=22
                  

### Breadth-First Search, LB=T

In [ ]:
import heapq

# --- 문제 데이터 및 헬퍼 함수 (이전과 동일) ---
jobs = [1, 2, 3, 4, 5]
processing = {1: 4, 2: 3, 3: 7, 4: 2, 5: 2}
due = {1: 5, 2: 6, 3: 8, 4: 8, 5: 17}

def complement(S):
    return [j for j in jobs if j not in S]

def P(S):
    return sum(processing[i] for i in S)

def true_cost(order):
    C = 0
    total_tardiness = 0
    for j in order:
        C += processing[j]
        total_tardiness += max(0, C - due[j])
    return total_tardiness

def LB1_suffix(S):
    S_bar = complement(S)
    C = P(S_bar)
    lb = 0
    for j in reversed(S):
        C += processing[j]
        lb += max(0, C - due[j])
    return lb

def LB2_suffix(S):
    base = LB1_suffix(S)
    S_bar = complement(S)
    if not S_bar:
        return base
    extra = min(max(0, P(S_bar) - due[k]) for k in S_bar)
    return base + extra

# --- Best-First Search (Incumbent Solution 갱신 과정 포함) ---
def breadth_first_search_with_incumbent_tracking(mode="LB1"):
    if mode == "LB1":
        lb_func = LB1_suffix
        print("\n=== (c) Best-First Search with LB(S) = Σ T_i ===")
    else:
        lb_func = LB2_suffix
        print(f"\n=== (d) Best-First Search with LB(S) = Σ T_i + min[max(0,P(S')-d_k)] ===")

    IC = float("inf")
    best_order = None # Incumbent Solution
    step = 0
    
    pq = [(0, tuple())] 

    while pq:
        current_lb, S_tuple = heapq.heappop(pq)
        S = list(S_tuple)
        
        print(f"\n[Step {step+1}] --------------------------------")
        # ✨ 현재 IC와 최적해 상태 출력
        print(f"  - 현재 IC (최적해 비용): {IC} | 현재 최적해 순서: {best_order}")
        print(f"  - 후보군에서 노드 선택: {str(S):<20} | LB = {current_lb}")
        
        if current_lb >= IC:
            print(f"  - 가지치기 (LB {current_lb} >= IC {IC})")
            step += 1
            continue
            
        print("  - 노드 확장:")
        for j in complement(S):
            step += 1
            new_S = S + [j]
            new_lb = lb_func(new_S)
            
            print(f"    [Step {step}] -> '{j}' 추가 -> {str(new_S):<20} | LB = {new_lb}", end="")

            if new_lb >= IC:
                print(f" (가지치기: LB >= IC {IC})")
                continue

            if len(new_S) == len(jobs):
                order = list(reversed(new_S))
                cost = true_cost(order)
                print(f" (완전해, 실제 비용 = {cost})")
                if cost < IC:
                    print(f"      ✨ IC 갱신! {IC} -> {cost}. 최적해 변경: {best_order} -> {order}")
                    IC = cost
                    best_order = order
            else:
                heapq.heappush(pq, (new_lb, tuple(new_S)))
                print(" (후보군에 추가)")
    
    print("\n======================================")
    print("탐색 종료!")
    print(f"최종 최적해 (Incumbent Solution): {best_order}")
    print(f"최소 총 지연시간 (Final IC): {IC}")
    print(f"총 스텝(LB 계산 횟수): {step}")
    print("======================================")


if __name__ == "__main__":
    breadth_first_search_with_incumbent_tracking(mode="LB1")


=== (c) Best-First Search with LB(S) = Σ T_i ===

[Step 1] --------------------------------
  - 현재 IC (최적해 비용): inf | 현재 최적해 순서: None
  - 후보군에서 노드 선택: []                   | LB = 0
  - 노드 확장:
    [Step 1] -> '1' 추가 -> [1]                  | LB = 13 (후보군에 추가)
    [Step 2] -> '2' 추가 -> [2]                  | LB = 12 (후보군에 추가)
    [Step 3] -> '3' 추가 -> [3]                  | LB = 10 (후보군에 추가)
    [Step 4] -> '4' 추가 -> [4]                  | LB = 10 (후보군에 추가)
    [Step 5] -> '5' 추가 -> [5]                  | LB = 1 (후보군에 추가)

[Step 6] --------------------------------
  - 현재 IC (최적해 비용): inf | 현재 최적해 순서: None
  - 후보군에서 노드 선택: [5]                  | LB = 1
  - 노드 확장:
    [Step 6] -> '1' 추가 -> [5, 1]               | LB = 12 (후보군에 추가)
    [Step 7] -> '2' 추가 -> [5, 2]               | LB = 11 (후보군에 추가)
    [Step 8] -> '3' 추가 -> [5, 3]               | LB = 9 (후보군에 추가)
    [Step 9] -> '4' 추가 -> [5, 4]               | LB = 9 (후보군에 추가)

[Step 10] --------------------------------
  - 현재 IC (최적해 비용): 

### Breadth-First Search, LB=T+min

In [50]:
import heapq

# --- 문제 데이터 및 헬퍼 함수 (이전과 동일) ---
jobs = [1, 2, 3, 4, 5]
processing = {1: 4, 2: 3, 3: 7, 4: 2, 5: 2}
due = {1: 5, 2: 6, 3: 8, 4: 8, 5: 17}

def complement(S):
    return [j for j in jobs if j not in S]

def P(S):
    return sum(processing[i] for i in S)

def true_cost(order):
    C = 0
    total_tardiness = 0
    for j in order:
        C += processing[j]
        total_tardiness += max(0, C - due[j])
    return total_tardiness

def LB1_suffix(S):
    S_bar = complement(S)
    C = P(S_bar)
    lb = 0
    for j in reversed(S):
        C += processing[j]
        lb += max(0, C - due[j])
    return lb

def LB2_suffix(S):
    base = LB1_suffix(S)
    S_bar = complement(S)
    if not S_bar:
        return base
    extra = min(max(0, P(S_bar) - due[k]) for k in S_bar)
    return base + extra


def breadth_first_search_with_incumbent_tracking(mode="LB1"):
    if mode == "LB1":
        lb_func = LB1_suffix
        print("\n=== (c) Breadth-First Search with LB(S) = Σ T_i ===")
    else:
        lb_func = LB2_suffix
        print(f"\n=== (d) Breadth-First Search with LB(S) = Σ T_i + min[max(0,P(S')-d_k)] ===")

    IC = float("inf")
    best_order = None # Incumbent Solution
    step = 0
    
    pq = [(0, tuple())] 

    while pq:
        current_lb, S_tuple = heapq.heappop(pq)
        S = list(S_tuple)
        
        print(f"\n[Step {step+1}] --------------------------------")
        # ✨ 현재 IC와 최적해 상태 출력
        print(f"  - 현재 IC (최적해 비용): {IC} | 현재 최적해 순서: {best_order}")
        print(f"  - 후보군에서 노드 선택: {str(S):<20} | LB = {current_lb}")
        
        if current_lb >= IC:
            print(f"  - 가지치기 (LB {current_lb} >= IC {IC})")
            step += 1
            continue
            
        print("  - 노드 확장:")
        for j in complement(S):
            step += 1
            new_S = S + [j]
            new_lb = lb_func(new_S)
            
            print(f"    [Step {step}] -> '{j}' 추가 -> {str(new_S):<20} | LB = {new_lb}", end="")

            if new_lb >= IC:
                print(f" (가지치기: LB >= IC {IC})")
                continue

            if len(new_S) == len(jobs):
                order = list(reversed(new_S))
                cost = true_cost(order)
                print(f" (완전해, 실제 비용 = {cost})")
                if cost < IC:
                    print(f"      ### IC 갱신! {IC} -> {cost}. 최적해 변경: {best_order} -> {order}")
                    IC = cost
                    best_order = order
            else:
                heapq.heappush(pq, (new_lb, tuple(new_S)))
                print(" (후보군에 추가)")
    
    print("\n======================================")
    print("탐색 종료!")
    print(f"최종 최적해 (Incumbent Solution): {best_order}")
    print(f"최소 총 지연시간 (Final IC): {IC}")
    print(f"총 스텝(LB 계산 횟수): {step}")
    print("======================================")


if __name__ == "__main__":
    breadth_first_search_with_incumbent_tracking(mode="LB2")


=== (d) Breadth-First Search with LB(S) = Σ T_i + min[max(0,P(S')-d_k)] ===

[Step 1] --------------------------------
  - 현재 IC (최적해 비용): inf | 현재 최적해 순서: None
  - 후보군에서 노드 선택: []                   | LB = 0
  - 노드 확장:
    [Step 1] -> '1' 추가 -> [1]                  | LB = 13 (후보군에 추가)
    [Step 2] -> '2' 추가 -> [2]                  | LB = 12 (후보군에 추가)
    [Step 3] -> '3' 추가 -> [3]                  | LB = 10 (후보군에 추가)
    [Step 4] -> '4' 추가 -> [4]                  | LB = 10 (후보군에 추가)
    [Step 5] -> '5' 추가 -> [5]                  | LB = 9 (후보군에 추가)

[Step 6] --------------------------------
  - 현재 IC (최적해 비용): inf | 현재 최적해 순서: None
  - 후보군에서 노드 선택: [5]                  | LB = 9
  - 노드 확장:
    [Step 6] -> '1' 추가 -> [5, 1]               | LB = 16 (후보군에 추가)
    [Step 7] -> '2' 추가 -> [5, 2]               | LB = 16 (후보군에 추가)
    [Step 8] -> '3' 추가 -> [5, 3]               | LB = 10 (후보군에 추가)
    [Step 9] -> '4' 추가 -> [5, 4]               | LB = 15 (후보군에 추가)

[Step 10] ------------------------